# Tugas Hands-On 3: Pandas Lanjutan
**Kasus 02: Customer Churn Analysis (Telekomunikasi)**, klasifikasi biner dengan target `status_pelanggan` (Aktif/Churn).

Dataset diunduh otomatis dari Google Drive. Jalankan dengan *Runtime → Run all*.

## 0. Setup

In [ ]:
import os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gdown

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 2026
np.random.seed(SEED)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

BASE = 'tugas-hands-on-3'
PATH = f'{BASE}/data/case2_customer_churn.csv'
os.makedirs(f'{BASE}/data', exist_ok=True)
os.makedirs(f'{BASE}/output', exist_ok=True)

FILE_ID = '1gu2dNvQXod2On75oIpceK8DoO0wy7tuI'
if not os.path.exists(PATH):
    gdown.download(id=FILE_ID, output=PATH, quiet=True)

## 1. Data

In [ ]:
df_mentah = pd.read_csv(PATH)
print('shape:', df_mentah.shape)
df_mentah.head()

In [ ]:
df_mentah.info()

In [ ]:
df_mentah.describe(include='all').T

In [ ]:
KOLOM_TARGET = 'status_pelanggan'
KOLOM_KUNCI  = 'customer_id'
KOLOM_GRUP   = 'jenis_kontrak'
KOLOM_NUM    = 'total_tagihan_idr'
KOLOM_LAMA, KOLOM_BULANAN = 'lama_berlangganan_bulan', 'tagihan_bulanan_idr'
KOLOM_BIN    = 'jumlah_komplain'

## 2. Kualitas Data
Audit dengan `cek_kualitas`, lalu bersihkan dengan urutan: duplikat → missing value → outlier → tipe data.

In [ ]:
def cek_kualitas(d, kolom_num, kolom_grup, kunci):
    g  = d.groupby(kolom_grup, observed=True)[kolom_num]
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    outlier = (d[kolom_num] < q1 - 1.5 * iqr) | (d[kolom_num] > q3 + 1.5 * iqr)
    return pd.Series({
        'n_baris':     len(d),
        'n_duplikat':  d.duplicated(subset=kunci).sum(),
        'n_missing':   d.isnull().sum().sum(),
        'n_outlier':   outlier.sum(),
    })

cek_kualitas(df_mentah, KOLOM_NUM, KOLOM_GRUP, KOLOM_KUNCI)

In [ ]:
print('Missing per kolom:')
print(df_mentah.isnull().sum()[lambda s: s > 0])
print('\nNilai unik jenis_kontrak:', df_mentah[KOLOM_GRUP].nunique())

### 2.0 Normalisasi label
`jenis_kontrak` punya 12 nilai padahal seharusnya 3 (Bulanan, Tahunan, Dua Tahun): ada variasi huruf besar/kecil dan karakter rusak seperti `T@hun@n`. **Alasan:** label yang sama tapi tertulis berbeda dianggap grup berbeda, sehingga `groupby`, outlier per grup, dan encoding menjadi salah.

In [ ]:
def samakan_label(s):
    kunci = s.str.strip().str.lower().str.replace('@', 'a', regex=False)
    kanon = s.groupby(kunci).agg(lambda x: x.value_counts().idxmax())
    return kunci.map(kanon).where(s.notna())

df_bersih = df_mentah.copy()
for c in df_bersih.select_dtypes(include=['object', 'string']).columns:
    if c in (KOLOM_TARGET, KOLOM_KUNCI):
        continue
    sebelum = df_bersih[c].nunique()
    df_bersih[c] = samakan_label(df_bersih[c])
    if df_bersih[c].nunique() != sebelum:
        print(f'{c}: {sebelum} -> {df_bersih[c].nunique()} nilai unik')

df_bersih[KOLOM_GRUP].value_counts(dropna=False)

### 2a. Duplikat
**Alasan:** satu pelanggan harus muncul satu kali. Baris duplikat dibuang agar bobot pelanggan tidak ganda.

In [ ]:
n0 = len(df_bersih)
df_bersih = df_bersih.drop_duplicates(subset=KOLOM_KUNCI, keep='first').reset_index(drop=True)
print(f'Duplikat dibuang: {n0 - len(df_bersih)} baris')

### 2b. Missing value
**Alasan:**
- `total_tagihan_idr` kosong diisi `lama berlangganan × tagihan bulanan` (rasio total / (lama × bulanan) rata-rata ≈ 1,00). Rumus memakai baris itu sendiri, jadi tidak ada leakage.
- Baris dengan target kosong dibuang (tidak bisa dipakai belajar) dan kolom dengan missing > 50% dibuang (informasi terlalu sedikit).
- Kategorik kosong diisi `'Unknown'` (nilai konstan, tanpa leakage).
- Numerik lain **tidak diisi di sini**; imputasi median dilakukan di pipeline setelah split.

In [ ]:
df_bersih = df_bersih.replace(r'^\s*$', np.nan, regex=True)

mask = df_bersih[KOLOM_NUM].isna() & df_bersih[KOLOM_LAMA].notna() & df_bersih[KOLOM_BULANAN].notna()
df_bersih.loc[mask, KOLOM_NUM] = df_bersih.loc[mask, KOLOM_LAMA] * df_bersih.loc[mask, KOLOM_BULANAN]
print(f'{KOLOM_NUM} diisi dari rumus: {mask.sum()} baris')

n0 = len(df_bersih)
df_bersih = df_bersih.dropna(subset=[KOLOM_TARGET]).reset_index(drop=True)
print(f'Baris target kosong dibuang: {n0 - len(df_bersih)}')

rasio = df_bersih.isnull().mean()
buang = [c for c in rasio[rasio > 0.5].index if c not in (KOLOM_TARGET, KOLOM_KUNCI)]
df_bersih = df_bersih.drop(columns=buang)
print('Kolom dibuang (>50% missing):', buang)

for c in df_bersih.select_dtypes(include=['object', 'string']).columns:
    if c not in (KOLOM_TARGET, KOLOM_KUNCI):
        df_bersih[c] = df_bersih[c].fillna('Unknown')

print('\nMissing tersisa (numerik, diimputasi setelah split):')
print(df_bersih.isnull().sum()[lambda s: s > 0])

### 2c. Outlier per grup
**Alasan:** sebaran tagihan tiap jenis kontrak berbeda, jadi batas IQR dihitung per grup. Nilai ekstrem di-*clip* (bukan dibuang) agar jumlah baris dan pelanggan churn tidak berkurang.

In [ ]:
g  = df_bersih.groupby(KOLOM_GRUP, observed=True)[KOLOM_NUM]
q1 = g.transform(lambda s: s.quantile(0.25))
q3 = g.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr

n_out = ((df_bersih[KOLOM_NUM] < bawah) | (df_bersih[KOLOM_NUM] > atas)).sum()
df_bersih[KOLOM_NUM] = df_bersih[KOLOM_NUM].clip(lower=bawah, upper=atas)
print(f'Outlier di-clip pada {KOLOM_NUM}: {n_out} baris')

### 2d. Tipe data
**Alasan:** target diubah ke 0/1 (Churn = 1) agar siap untuk model; teks bernilai sedikit diubah ke `category`.

In [ ]:
df_bersih[KOLOM_TARGET] = (df_bersih[KOLOM_TARGET] == 'Churn').astype(int)

for c in df_bersih.select_dtypes(include=['object', 'string']).columns:
    if c != KOLOM_KUNCI:
        df_bersih[c] = df_bersih[c].astype('category')

df_bersih.dtypes

### Tabel kualitas sebelum vs sesudah

In [ ]:
laporan = pd.DataFrame({
    'sebelum': cek_kualitas(df_mentah, KOLOM_NUM, KOLOM_GRUP, KOLOM_KUNCI),
    'sesudah': cek_kualitas(df_bersih, KOLOM_NUM, KOLOM_GRUP, KOLOM_KUNCI),
})
laporan.to_csv(f'{BASE}/laporan_kualitas.csv')
laporan

**Interpretasi:**
- `n_duplikat` tetap 0: `customer_id` memang sudah unik, tidak ada baris yang dibuang.
- `n_missing` turun dari 1.638 menjadi 803. Penurunan berasal dari kategorik yang diisi `'Unknown'` dan total tagihan yang dihitung dari rumus. Sisa 803 hanya di kolom numerik dan sengaja diimputasi setelah split.
- `n_outlier` turun dari 65 menjadi 0 karena nilai ekstrem di-*clip* per jenis kontrak. Normalisasi label (12 → 3 nilai) juga membuat grup pembanding menjadi benar.

## 3. Analisis
`groupby` lanjutan, kuartil, dan `pivot_table`. Dataset tidak punya kolom tanggal, sehingga tren waktu dilewati. RFM juga dilewati karena data berisi satu baris profil per pelanggan, bukan riwayat transaksi.

In [ ]:
print(f'Tingkat churn keseluruhan: {df_bersih[KOLOM_TARGET].mean():.1%}')

ringkas = (df_bersih.groupby(KOLOM_GRUP, observed=True)
           .agg(n=(KOLOM_TARGET, 'size'),
                churn_rate=(KOLOM_TARGET, 'mean'),
                rata_num=(KOLOM_NUM, 'mean'))
           .sort_values('churn_rate', ascending=False))
ringkas

In [ ]:
kuartil = pd.qcut(df_bersih[KOLOM_BIN], 4, duplicates='drop')
per_kuartil = df_bersih.groupby(kuartil, observed=True)[KOLOM_TARGET].agg(['size', 'mean'])
display(per_kuartil)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=ringkas.index.astype(str), y=ringkas['churn_rate'], ax=ax[0])
ax[0].set_title(f'Churn rate per {KOLOM_GRUP}')
sns.barplot(x=per_kuartil.index.astype(str), y=per_kuartil['mean'], ax=ax[1])
ax[1].set_title(f'Churn rate per kuartil {KOLOM_BIN}')
ax[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
df_bersih.groupby('jumlah_layanan_aktif')[KOLOM_TARGET].agg(['size', 'mean']).round(3)

In [ ]:
pivot = df_bersih.pivot_table(index=KOLOM_GRUP, columns='layanan_internet',
                              values=KOLOM_TARGET, aggfunc='mean', observed=True)
pivot.style.format('{:.1%}').background_gradient(cmap='Reds', axis=None)

In [ ]:
df_bersih.select_dtypes('number').corrwith(df_bersih[KOLOM_TARGET]).drop(KOLOM_TARGET).sort_values(key=abs, ascending=False)

### 3 Temuan yang bisa ditindaklanjuti
Tingkat churn keseluruhan adalah **43,6%**.

1. **Kontrak bulanan churn 59,7%, jauh di atas kontrak tahunan (39,7%) dan dua tahun (31,3%)** → tawarkan insentif migrasi ke kontrak jangka panjang (diskon atau bonus layanan).
2. **Churn naik seiring jumlah komplain: 23,9% pada ≤3 komplain menjadi sekitar 57% pada >8 komplain** → jadikan komplain > 8 sebagai peringatan dini dan prioritaskan penanganannya.
3. **Pelanggan dengan ≤3 layanan aktif churn 49,5% vs sekitar 40% pada ≥4 layanan** → dorong *bundling* atau *cross-sell* agar pelanggan lebih terikat.

## 4. Persiapan Model
Split dulu (dengan `stratify=y`), baru imputasi, scaling, dan encoding lewat `ColumnTransformer` yang di-`fit` hanya pada data train.

In [ ]:
X = df_bersih.drop(columns=[KOLOM_TARGET, KOLOM_KUNCI])
y = df_bersih[KOLOM_TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Proporsi churn -> train: %.3f | test: %.3f' % (y_train.mean(), y_test.mean()))

In [ ]:
num_cols = X_train.select_dtypes('number').columns.tolist()
cat_cols = X_train.select_dtypes(exclude='number').columns.tolist()

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols),
])

X_train_p = prep.fit_transform(X_train)
X_test_p  = prep.transform(X_test)

print('Setelah preprocessing -> train:', X_train_p.shape, '| test:', X_test_p.shape)

**Bukti tidak ada leakage:** statistik imputer dan scaler dihitung hanya dari data train, lalu dipakai apa adanya untuk data test. Batas *clip* outlier dihitung di tahap pembersihan, sebelum split, sesuai urutan tugas; pengaruhnya sangat kecil karena hanya membatasi nilai ekstrem.

In [ ]:
imp = prep.named_transformers_['num'].named_steps['imp']
sc  = prep.named_transformers_['num'].named_steps['sc']

assert np.allclose(imp.statistics_, X_train[num_cols].median().values)
assert set(X_train.index).isdisjoint(X_test.index)

pd.DataFrame({
    'median_imputer': imp.statistics_,
    'median_train':   X_train[num_cols].median().values,
    'mean_scaler':    sc.mean_,
}, index=num_cols).round(2)

In [ ]:
df_bersih.to_csv(f'{BASE}/output/data_bersih.csv', index=False)

kamus = pd.DataFrame({
    'kolom':     df_bersih.columns,
    'tipe':      df_bersih.dtypes.astype(str).values,
    'n_missing': df_bersih.isnull().sum().values,
    'n_unik':    df_bersih.nunique().values,
})
kamus.to_csv(f'{BASE}/output/kamus_data.csv', index=False)
kamus

## 5. README

In [ ]:
readme = f'''# Tugas Hands-On 3: Pandas Lanjutan

Kasus: 02 Customer Churn Analysis (Telekomunikasi), klasifikasi biner.

## Temuan
1. Kontrak bulanan churn 59,7% vs tahunan 39,7% dan dua tahun 31,3% -> dorong migrasi kontrak.
2. Churn naik seiring komplain (23,9% pada <=3 komplain, sekitar 57% pada >8) -> peringatan dini.
3. Pelanggan <=3 layanan aktif churn 49,5% vs sekitar 40% pada >=4 layanan -> bundling/cross-sell.

## Cara menjalankan
Buka notebook.ipynb di Google Colab, lalu Runtime -> Run all. Dataset diunduh otomatis dari Google Drive.
'''
with open(f'{BASE}/README.md', 'w') as f:
    f.write(readme)

### Cek file hasil

In [ ]:
for f in ['README.md', 'laporan_kualitas.csv', 'output/data_bersih.csv', 'output/kamus_data.csv']:
    path = f'{BASE}/{f}'
    print('OK    ' if os.path.exists(path) else 'HILANG', path)